[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caracena/apunte-analitica-textual/blob/main/capitulos/04_clasificacion.ipynb)

# Capítulo 4. Clasificación de textos

Este notebook forma parte del apunte del curso **Analítica Textual** y está preparado para ejecutarse en Google Colab o en Jupyter local.


In [ ]:
# Setup opcional para Colab
# En general, este notebook corre sin instalaciones adicionales.
# Si tu entorno no tiene las librerías base, descomenta la línea siguiente.
# !pip -q install scikit-learn pandas


## Objetivos

En este capítulo pasamos a aprendizaje supervisado: ahora sí tenemos etiquetas y queremos predecirlas.

Al finalizar deberías poder:

- formular un problema de clasificación de textos;
- entrenar un pipeline con TF-IDF y un clasificador;
- comparar modelos como Naive Bayes y SVM;
- evaluar resultados con métricas adecuadas.

## 4.1 Casos típicos

La clasificación textual aparece en:

- detección de spam;
- categorización de tickets;
- análisis de sentimiento;
- ruteo automático de documentos;
- clasificación temática de noticias.

## 4.2 Dataset de ejemplo


In [ ]:
import pandas as pd

datos = pd.DataFrame({
    "texto": [
        "el paciente requiere control médico y exámenes",
        "la clínica agenda consulta y seguimiento",
        "sube el precio del dólar y cae la bolsa",
        "el banco ajusta su proyección de inflación",
        "el delantero marcó dos goles en el torneo",
        "el campeonato terminó con una final intensa",
        "nueva cobertura médica para adultos mayores",
        "mercado financiero anticipa recorte de tasas",
        "el entrenador confirmó la formación titular"
    ],
    "etiqueta": [
        "salud", "salud",
        "finanzas", "finanzas",
        "deportes", "deportes",
        "salud", "finanzas", "deportes"
    ]
})

datos


## 4.3 Separación entrenamiento y prueba


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    datos["texto"],
    datos["etiqueta"],
    test_size=0.33,
    random_state=42,
    stratify=datos["etiqueta"]
)


## 4.4 Pipeline con SVM lineal


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

modelo_svm = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LinearSVC())
])

modelo_svm.fit(X_train, y_train)
pred_svm = modelo_svm.predict(X_test)
pred_svm


## 4.5 Pipeline con Naive Bayes


In [ ]:
from sklearn.naive_bayes import MultinomialNB

modelo_nb = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])

modelo_nb.fit(X_train, y_train)
pred_nb = modelo_nb.predict(X_test)
pred_nb


## 4.6 Evaluación


In [ ]:
from sklearn.metrics import classification_report

print("SVM")
print(classification_report(y_test, pred_svm))

print("Naive Bayes")
print(classification_report(y_test, pred_nb))


La lectura de métricas debe conectarse con el contexto:

- `accuracy` resume aciertos globales;
- `precision` importa si un falso positivo es costoso;
- `recall` importa si perder casos relevantes es grave;
- `F1` equilibra precision y recall.

## 4.7 ¿Cuándo elegir cada modelo?

`MultinomialNB`:

- rápido;
- fuerte como línea base;
- especialmente útil con bolsas de palabras.

`LinearSVC`:

- suele rendir muy bien en alta dimensión;
- tolera fronteras complejas mejor que NB;
- es una referencia sólida para texto clásico.

Redes neuronales:

- pueden superar enfoques lineales, sobre todo con más datos;
- requieren mayor costo computacional y diseño.

## 4.8 Interpretabilidad

En problemas de negocio conviene inspeccionar qué términos empujan una predicción. Con modelos lineales eso es más accesible que con arquitecturas profundas.

## 4.9 Riesgos frecuentes

- clases desbalanceadas;
- fuga de información entre train y test;
- etiquetas inconsistentes;
- métricas correctas sobre datos irreales.

## Ejercicios

1. Amplía el dataset con 5 textos por categoría y vuelve a entrenar.
2. Prueba `ngram_range=(1, 2)` en `TfidfVectorizer`.
3. Construye una matriz de confusión e interpreta los errores.

## Idea clave

En clasificación de textos, un pipeline simple con TF-IDF y SVM puede ser sorprendentemente competitivo. La calidad de etiquetas y la evaluación suelen importar más que la complejidad del algoritmo.
